# RQ3

## Setup

* Follow the instructions on ```Readme.md```
* The classification threshold can be changed in ```src/config.py```
* Inside the *pipeline* folder, run ```python -m src.rq3``` for testing all models OR 
```python -m src.rq3 --model_name``` for a specific model
    * Ex: ```python -m src.rq3 --model_name microsoft/codebert-base```
* For running the default models (without finetuning) add ```-default``` to the model name
    * Ex: ```python -m src.rq3 --model_name microsoft/codebert-base-default```
* Tested models: ```'microsoft/codebert-base', 'Salesforce/codet5-base'```
* Supported languages: ```'python', 'java' , 'cs', 'c'```
* Results are saved on ```results/RQ3```

## Results

In [8]:
import pandas as pd
import numpy as np

# Load CSV
csv_path = "../results/RQ3/clone_detection.csv"
df = pd.read_csv(csv_path)

THRESHOLD = 0.6        # set to None to disable filtering
SORT_BY = "f1"         # precision | recall | f1 | mcc | None
DESCENDING = True

if THRESHOLD is not None:
    df = df[np.isclose(df["threshold"], THRESHOLD)]

df = df[~df["model"].str.contains("-default", na=False)]

df = df[df["test_dataset"] == "SemanticCloneBench"]

if SORT_BY is not None:
    df = df.sort_values(by=SORT_BY, ascending=not DESCENDING)


display(
    df[[
        "model",
        "train_dataset",
        "test_dataset",
        "lan",
        "pairs",
        "threshold",
        "precision",
        "recall",
        "f1",
        "mcc",
        "TP",
        "TN",
        "FP",
        "FN",
    ]].round(4).reset_index(drop=True)
)

,model,train_dataset,test_dataset,lan,pairs,threshold,precision,recall,f1,mcc,TP,TN,FP,FN
0,Salesforce/codet5-base,Kamino,SemanticCloneBench,python,2000,0.6,0.8969,0.7480,0.8157,0.6713,748,914,86,252
1,Salesforce/codet5-base,Kamino,SemanticCloneBench,csharp,1870,0.6,0.9809,0.6588,0.7882,0.6839,616,923,12,319
2,microsoft/codebert-base,Kamino,SemanticCloneBench,python,2000,0.6,0.9727,0.6410,0.7728,0.6627,641,982,18,359
3,Salesforce/codet5-base,Kamino,SemanticCloneBench,java,1992,0.6,0.9455,0.6446,0.7666,0.6407,642,959,37,354
4,microsoft/codebert-base,Kamino,SemanticCloneBench,csharp,1870,0.6,0.9947,0.6075,0.7543,0.6560,568,932,3,367
5,microsoft/codebert-base,Kamino,SemanticCloneBench,java,1992,0.6,0.9832,0.5884,0.7362,0.6315,586,986,10,410


In [ ]:
import pandas as pd

# Load CSV
df = pd.read_csv(csv_path)

# Add type column
df['type'] = df['model'].apply(lambda x: 'pretrained' if '-default' in x else 'finetuned')

# Simplify model names
df['base_model'] = df['model'].apply(lambda x: x.split('/')[-1].replace('-default','').replace('-base',''))

# Sort by dataset, pairs descending
df = df.sort_values(by=['dataset','pairs'], ascending=[True, False])

# Column definition
col_def = "L{1.5cm}L{1.5cm}L{0.7cm}R{1.2cm}|R{0.8cm}R{0.8cm}R{0.8cm}R{0.8cm}|R{0.8cm}R{0.8cm}R{0.8cm}R{0.8cm}"

print("\\begin{table}[ht]")
print("\\centering")
print("\\footnotesize")
print("\\addtolength{\\tabcolsep}{-2pt}")
print("\\caption{Results for RQ3}")
print(f"\\begin{{tabular}}{{{col_def}}}")
print("\\toprule")
# Header
print("\\multirow{2}{*}{\\textbf{Model}} & \\multirow{2}{*}{\\textbf{Dataset}} & \\multirow{2}{*}{\\textbf{Lang}} & \\multirow{2}{*}{\\textbf{Pairs}} & " +
      "\\multicolumn{4}{c|}{\\textbf{Pretrained}} & \\multicolumn{4}{c}{\\textbf{Finetuned}} \\\\")
print("& & & & \\textbf{Prec.} & \\textbf{Rec.} & \\textbf{F1} & \\textbf{MCC} & \\textbf{Prec.} & \\textbf{Rec.} & \\textbf{F1} & \\textbf{MCC} \\\\")
print("\\midrule")

# Loop through models
for bm in df['base_model'].unique():
    df_bm = df[df['base_model']==bm]
    model_rows = len(df_bm['lan'].unique())  # number of language groups
    
    model_first = True
    for ds in df_bm['dataset'].unique():
        df_ds = df_bm[df_bm['dataset']==ds]
        dataset_rows = len(df_ds['lan'].unique())
        dataset_first = True
        
        for lang in df_ds['lan'].unique():
            df_lang = df_ds[df_ds['lan']==lang].sort_values(by='pairs', ascending=False).iloc[0]
            
            pre_row = df_ds[(df_ds['lan']==lang) & (df_ds['type']=='pretrained')].iloc[0]
            fin_row = df_ds[(df_ds['lan']==lang) & (df_ds['type']=='finetuned')].iloc[0]
            
            model_cell = f"\\multirow{{{model_rows}}}{{*}}{{{bm}}}" if model_first else ""
            dataset_cell = f"\\multirow{{{dataset_rows}}}{{*}}{{{ds}}}" if dataset_first else ""
            
            line = f"{model_cell} & {dataset_cell} & {lang} & {pre_row['pairs']} & " \
                   f"{pre_row['precision']:.2f} & {pre_row['recall']:.2f} & {pre_row['f1']:.2f} & {pre_row['mcc']:.2f} & " \
                   f"{fin_row['precision']:.2f} & {fin_row['recall']:.2f} & {fin_row['f1']:.2f} & {fin_row['mcc']:.2f} \\\\"
            print(line)
            
            model_first = False
            dataset_first = False
    print("\\midrule")

print("\\bottomrule")
print("\\end{tabular}")
print("\\label{tab:rq3Results}")
print("\\end{table}")


\begin{table}[ht]
\footnotesize
\addtolength{\tabcolsep}{-2pt}
\caption{Results for RQ3}
\begin{tabular}{L{1.2cm}L{1.5cm}L{1cm}R{1.2cm}|R{0.8cm}R{0.8cm}R{0.8cm}R{0.8cm}|R{0.8cm}R{0.8cm}R{0.8cm}R{0.8cm}}
\toprule
\multirow{2}{*}{\textbf{Model}} & \multirow{2}{*}{\textbf{Dataset}} & \multirow{2}{*}{\textbf{Lang}} & \multirow{2}{*}{\textbf{Pairs}} & \multicolumn{4}{c|}{\textbf{Pretrained}} & \multicolumn{4}{c}{\textbf{Finetuned}} \\
& & & & \textbf{Prec.} & \textbf{Rec.} & \textbf{F1} & \textbf{MCC} & \textbf{Prec.} & \textbf{Rec.} & \textbf{F1} & \textbf{MCC} \\
\midrule
\multirow{4}{*}{codebert} & \multirow{4}{*}{GPTCloneBench} & java & 14192 & 0.50 & 1.00 & 0.67 & -0.01 & 0.98 & 0.82 & 0.89 & 0.81 \\
 &  & csharp & 9816 & 0.50 & 1.00 & 0.67 & -0.01 & 1.00 & 0.89 & 0.94 & 0.89 \\
 &  & c & 6890 & 0.50 & 1.00 & 0.67 & 0.06 & 0.97 & 0.48 & 0.64 & 0.54 \\
 &  & python & 5640 & 0.50 & 1.00 & 0.67 & 0.00 & 0.98 & 0.81 & 0.89 & 0.81 \\
 & \multirow{1}{*}{Kamino} & python & 21172 & 0.52 & 0.97

In [ ]:
from datasets import load_from_disk

# Load your dataset from a local folder
ds = load_from_disk("../dataset/kamino_clones_dataset")

# If this is a DatasetDict (train/validation/test), you can inspect splits:
print(ds)

# Count number of items:
if isinstance(ds, dict):
    # DatasetDict case
    for split, subset in ds.items():
        print(split, len(subset))
else:
    # Single Dataset case
    print("Total elements:", len(ds))
